# Phase 1 EDA: CORD v2 train split
Run from the repo root after `pip install -e .[data,nb]`. Findings are written up in `explain/01-phase1-data.md`.

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2", split="train")
gts = [json.loads(s)["gt_parse"] for s in ds["ground_truth"]]

def items(g):
    m = g.get("menu", [])
    return len(m) if isinstance(m, list) else 1

n_items = Counter(items(g) for g in gts)
sizes = Counter(ex["image"].size for ex in ds)
def has(g, path):
    for k in path.split("/"):
        if not isinstance(g, dict) or k not in g:
            return False
        g = g[k]
    return True
fields = ["menu/nm", "menu/cnt", "menu/price", "menu/unitprice", "menu/sub", "sub_total/subtotal_price", "sub_total/tax_price",
          "sub_total/service_price", "sub_total/discount_price", "total/total_price", "total/cashprice", "total/changeprice"]
def first_item(g):
    m = g.get("menu", {})
    return m[0] if isinstance(m, list) and m else m if isinstance(m, dict) else {}
present = {}
for f in fields:
    if f.startswith("menu/"):
        k = f.split("/")[1]
        present[f] = sum(k in first_item(g) for g in gts) / len(gts)
    else:
        present[f] = sum(has(g, f) for g in gts) / len(gts)

In [ ]:
from dociq.data.cord import json2token

lens = sorted(len(json2token(g, set())) for g in gts)
tags = set()
for g in gts: json2token(g, tags)

In [ ]:
print(len(gts), 'receipts;', len(tags), 'distinct tags')
print('items per receipt', sorted(n_items.items())[:10])
print('image sizes', sizes.most_common(5))
for k, v in present.items(): print(f'{k:28s} {v:.2f}')
print('target chars median/p95/max', lens[len(lens)//2], lens[int(.95*len(lens))], lens[-1])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(*zip(*sorted(n_items.items())))
ax[0].set_title('Items per receipt')
ax[1].barh(list(present)[::-1], list(present.values())[::-1])
ax[1].set_title('Share of receipts with field')
plt.tight_layout()

## Look at 20 images by eye
Note blur, rotation, crumpled paper, handwriting: these decide what the robustness test should simulate.

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(15, 14))
for a, i in zip(axs.flat, range(20)):
    a.imshow(ds[i]['image']); a.axis('off'); a.set_title(str(i))
plt.tight_layout()